# Mean-head experiment - Kaggle: GPU T4 x2 + Internet ON

**Run All, walk away (~2 h on GPU).** Runs the whole pre-registered campaign end to end:

| stage | runs | what it does |
|---|---|---|
| 1 | 4 | lambda selection on the `through2016` fold (candidates 0.1 / 0.3 / 1.0 / 3.0) |
| 2 | 0 | scores them by 2016-validation PPR MAE and **freezes the winner into all nine mh configs** |
| 3 | 9 | v1 baseline backfill (`through2016/17/18` x seeds 42/43/44) |
| 4 | 27 | the mean-head main arm (9 folds x 3 seeds) at the frozen lambda |

Spec: `docs/superpowers/specs/2026-07-26-mean-head-design.md` (PRE-REGISTERED - the gate in section 8
is binding, and a failure is an honest negative, not a reason to retune).

**Why stage 2 is automated rather than hand-edited:** the spec requires lambda be selected on the 2016
validation season and then frozen identically across all nine folds. Doing that by hand across nine
files mid-session is exactly where a typo would silently invalidate 27 runs, so the notebook writes it
and commits the change as the audit trail.

Training is checkpoint-resumable and skip-if-complete, so a session cutoff loses nothing - just
Run All again. Folds are trained **fold-major (all 3 seeds together)** so a partial session still
yields whole, usable folds.

WARNING: the setup cell HARD-FAILS if the GPU is off. Set Session options -> Accelerator -> **GPU T4 x2**
(never P100 - it is sm_60 and this torch build has no kernels for it) before Run All.

In [ ]:
import os, pathlib, subprocess
root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if root is None:
    subprocess.run(["git", "clone", "https://github.com/mtsilverstein/Megatron.git"], check=True)
    root = pathlib.Path.cwd() / "Megatron"
os.chdir(root)
print("cwd:", os.getcwd())

!git pull
!pip install -q -e .
!python -m ffmodel.data.pull --seasons 2012 2025 --out data/raw
!python -c "from pathlib import Path; from ffmodel.data.pull import pull_weekly, pull_schedules; from ffmodel.data.features import build_features; s=list(range(2012,2026)); build_features(pull_weekly(s, Path('data/raw')), pull_schedules(s, Path('data/raw'))).to_parquet('data/features_2012_2025.parquet')"

import torch
print("cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "NO GPU. Set Session options -> Accelerator -> GPU T4 x2, then Run All again. "
    "(Two earlier campaigns silently CPU-trained only the fastest folds.)")
print("gpu:", torch.cuda.get_device_name(0))

# Kaggle gotcha: this kernel STARTED BEFORE `pip install -e .` ran, so its
# sys.path has no ffmodel. Subprocess calls (`python -m ffmodel...`) are fine
# because they are fresh processes -- only IN-KERNEL imports break. Put src/ on
# the path so later cells can import ffmodel directly.
import sys as _sys, pathlib as _pl
_src = str(_pl.Path.cwd() / "src")
if _src not in _sys.path:
    _sys.path.insert(0, _src)
print("sys.path bootstrapped for in-kernel imports:", _src)


## Stage 1 - lambda selection (4 runs, seed 42, `through2016` only)

Each candidate has its own `run_name` (`mh_l01/03/10/30`) so the four artifacts coexist. They
deliberately do NOT share `run_name: mh` - that would make each run delete the previous one's
artifact, leaving only the last candidate and no comparison.

In [ ]:
import subprocess

LAMBDA_CONFIGS = [
    ("configs/transformer_mh_lambda01_through2016.yaml", 0.1),
    ("configs/transformer_mh_lambda03_through2016.yaml", 0.3),
    ("configs/transformer_mh_lambda10_through2016.yaml", 1.0),
    ("configs/transformer_mh_lambda30_through2016.yaml", 3.0),
]
for cfg, lam in LAMBDA_CONFIGS:
    print("\n=== lambda selection: %s  (lambda=%s, seed 42) ===" % (cfg, lam), flush=True)
    subprocess.run(["python", "-m", "ffmodel.model.train", "--config", cfg,
                    "--features-parquet", "data/features_2012_2025.parquet"], check=True)
print("\nStage 1 complete: all four lambda candidates trained.")

## Stage 2 - score the candidates and freeze the winner

Scores each candidate by **2016-validation PPR MAE from the Arm-A point estimate** (the pre-registered
criterion), then writes the winning `mean_lambda` into all nine main-arm configs and commits it.

Note `data_dir="data/raw"` - that is where the setup cell pulled nflverse to, and it differs from the
module's own default of `data`.

In [ ]:
# Kaggle gotcha: this kernel STARTED BEFORE `pip install -e .` ran, so its
# sys.path has no ffmodel. Subprocess calls (`python -m ffmodel...`) are fine
# because they are fresh processes -- only IN-KERNEL imports break. Put src/ on
# the path so later cells can import ffmodel directly.
import sys as _sys, pathlib as _pl
_src = str(_pl.Path.cwd() / "src")
if _src not in _sys.path:
    _sys.path.insert(0, _src)
print("sys.path bootstrapped for in-kernel imports:", _src)

from pathlib import Path
import re, subprocess
import yaml
from ffmodel.model.select_lambda import select_lambda
from ffmodel.model.train import _validate_mean_cfg

ROOTS = [Path("models/transformer/mh_l%s" % s) for s in ("01", "03", "10", "30")]
table = select_lambda(ROOTS, val_season=2016, data_dir=Path("data/raw"))
print(table.to_string(index=False))

winner = float(table.iloc[0]["mean_lambda"])
print("\nSELECTED mean_lambda = %g  (ppr_mae %.4f, root %s)"
      % (winner, table.iloc[0]["ppr_mae"], table.iloc[0]["root"]))

# Freeze it into every main-arm config. Pre-registered: ONE lambda, all nine folds.
changed = []
for year in range(2016, 2025):
    p = Path("configs/transformer_mh_through%d.yaml" % year)
    s = p.read_text()
    new = re.sub(r"^mean_lambda:.*$", "mean_lambda: %g" % winner, s, count=1, flags=re.M)
    if new != s:
        p.write_text(new)
        changed.append(p.name)
print("froze lambda=%g into %d configs" % (winner, len(changed)))

# Sanity: all nine must now agree, and the config guard must accept them.
lams = set()
for year in range(2016, 2025):
    cfg = yaml.safe_load(Path("configs/transformer_mh_through%d.yaml" % year).read_text())
    _validate_mean_cfg(cfg)          # rejects lambda 0 / predict_mean without a loss term
    lams.add(cfg["mean_lambda"])
assert lams == {winner}, "configs disagree on lambda: %s" % lams
print("all nine main-arm configs validated and agree")

subprocess.run(["git", "add", "configs"], check=False)
subprocess.run(["git", "commit", "-m",
                "experiment: freeze mean_lambda=%g from through2016 selection" % winner],
               check=False)

## Stage 3 - v1 baseline backfill (9 runs)

The committed v1 baseline already covers `through2019`-`through2025`. These three folds are the ones
the 9-fold design adds; without them the earliest three folds have no baseline to compare against.

In [ ]:
import subprocess

for year in (2016, 2017, 2018):
    cfg = "configs/transformer_v1_through%d.yaml" % year
    for seed in (42, 43, 44):
        cmd = ["python", "-m", "ffmodel.model.train", "--config", cfg,
               "--features-parquet", "data/features_2012_2025.parquet"]
        if seed != 42:
            cmd += ["--seed", str(seed)]
        print("\n=== baseline %s  seed %d ===" % (cfg, seed), flush=True)
        subprocess.run(cmd, check=True)
    print("*** v1 through%d COMPLETE (3/3 seeds) ***" % year, flush=True)
print("\nStage 3 complete: baseline backfill done.")

## Stage 4 - mean-head main arm (27 runs)

Fold-major, so a session cutoff leaves whole folds usable. Runs at the lambda frozen in stage 2.

In [ ]:
import subprocess

for year in range(2016, 2025):
    cfg = "configs/transformer_mh_through%d.yaml" % year
    for seed in (42, 43, 44):
        cmd = ["python", "-m", "ffmodel.model.train", "--config", cfg,
               "--features-parquet", "data/features_2012_2025.parquet"]
        if seed != 42:
            cmd += ["--seed", str(seed)]
        print("\n=== main arm %s  seed %d ===" % (cfg, seed), flush=True)
        subprocess.run(cmd, check=True)
    print("*** mh through%d COMPLETE (3/3 seeds) ***" % year, flush=True)
print("\nStage 4 complete: main arm done.")

## Status - what is actually finished

Safe to run after a partial session; reports per-fold completeness for BOTH arms, since the gate needs
them paired.

In [ ]:
import json
from pathlib import Path

def complete(run, year):
    m = Path("models/transformer/%s/through%d/metrics.json" % (run, year))
    return m.exists() and json.loads(m.read_text()).get("complete") is True

print("fold          v1 baseline   mh arm")
ready = 0
for year in range(2016, 2025):
    v1 = sum(complete(r, year) for r in ("v1", "v1_s43", "v1_s44"))
    mh = sum(complete(r, year) for r in ("mh", "mh_s43", "mh_s44"))
    both = (v1 == 3 and mh == 3)
    ready += both
    print("through%d     %d/3           %d/3   %s"
          % (year, v1, mh, "<- PAIRED, testable" if both else ""))
print("\n%d/9 folds have BOTH arms complete." % ready)
print("The pre-registered gate wants all 9; fewer still gives a weaker paired comparison.")

## Push the artifacts back

Falls back to a zip if the push fails (a fresh Kaggle session has no git credentials).

In [ ]:
import subprocess, zipfile
from pathlib import Path

roots = (["models/transformer/mh%s" % s for s in ("", "_s43", "_s44")]
         + ["models/transformer/v1%s" % s for s in ("", "_s43", "_s44")]
         + ["models/transformer/mh_l%s" % s for s in ("01", "03", "10", "30")])
subprocess.run(["git", "add"] + roots, check=False)
subprocess.run(["git", "commit", "-m",
                "model: mean-head experiment arms + v1 backfill (9 folds x 3 seeds)"], check=False)
if subprocess.run(["git", "push"], check=False).returncode == 0:
    print("Pushed. NEXT: wire predict_point into the eval harness, then read the pre-registered gate.")
else:
    zp = Path("mean_head_artifacts.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for r in roots:
            for f in Path(r).rglob("through*/*"):
                if f.is_file():
                    zf.write(f, f.relative_to("."))
    print("git push failed -- download %s" % zp.resolve())